In [ ]:
import time
import importlib
import engines.llm
import engines.rag

# Eagerly reload modules to ensure fresh code in Colab memory
importlib.reload(engines.llm)
importlib.reload(engines.rag)

from engines.llm import load_llm_engine
from engines.rag import QueryIntent, classify_intent

# 1. Eagerly load Qwen3-35B engine onto GPU
print("⚙️ Pre-loading Qwen3-35B engine onto GPU (Non-Thinking Mode, max_tokens=4096)...")
load_llm_engine()
print("✅ Qwen3-35B engine ready.\n")

# 2. Define Benchmark Test Dataset
benchmark_queries = [
    # =========================================================================
    # --- METADATA_QUERY Benchmark (33 Queries) ---
    # =========================================================================
    ("List all certificates issued in Germany", QueryIntent.METADATA_QUERY),
    ("How many certificates expire in 2026?", QueryIntent.METADATA_QUERY),
    ("Which certificates are issued by TÜV SÜD?", QueryIntent.METADATA_QUERY),
    ("Show all certificates granted by the FCC", QueryIntent.METADATA_QUERY),
    ("Count the total number of RF certificates in the database", QueryIntent.METADATA_QUERY),
    ("List all certificates issued to Bosch", QueryIntent.METADATA_QUERY),
    ("Which certificates were approved between 2022 and 2024?", QueryIntent.METADATA_QUERY),
    ("Find the certificate ID for model number RF-7700", QueryIntent.METADATA_QUERY),
    ("How many certificates were issued by Anatel in Brazil?", QueryIntent.METADATA_QUERY),
    ("List all certificates expiring before December 2025", QueryIntent.METADATA_QUERY),
    ("Which certificates belong to Continental Automotive?", QueryIntent.METADATA_QUERY),
    ("Show all certificates issued by Dekra Testing Services", QueryIntent.METADATA_QUERY),
    ("Find all RF certificates registered under the manufacturer Denso", QueryIntent.METADATA_QUERY),
    ("How many certificates were approved in Japan?", QueryIntent.METADATA_QUERY),
    ("List all certificates with an expiration date in 2028", QueryIntent.METADATA_QUERY),
    ("Which certificates were issued by ISED Canada?", QueryIntent.METADATA_QUERY),
    ("Show the latest 5 certificates added to the system", QueryIntent.METADATA_QUERY),
    ("Find the issuing authority for certificate ID cert_9942a", QueryIntent.METADATA_QUERY),
    ("How many certificates are associated with 77GHz radar modules?", QueryIntent.METADATA_QUERY),
    ("List all active certificates issued in South Korea by KCC", QueryIntent.METADATA_QUERY),
    ("Which certificates were signed by Bureau Veritas?", QueryIntent.METADATA_QUERY),
    ("Show all certificates for model numbers starting with 'UWB-'", QueryIntent.METADATA_QUERY),
    ("Find all certificates issued in the United States", QueryIntent.METADATA_QUERY),
    ("How many certificates expire within the next 6 months?", QueryIntent.METADATA_QUERY),
    ("List all certificates granted to Valeo Mobility", QueryIntent.METADATA_QUERY),
    ("Which certificates were issued in 2021?", QueryIntent.METADATA_QUERY),
    ("Show all certificates managed by SGS North America", QueryIntent.METADATA_QUERY),
    ("Count how many certificates are registered under Aptiv", QueryIntent.METADATA_QUERY),
    ("List all certificates issued by TELEC Japan", QueryIntent.METADATA_QUERY),
    ("Find all certificates expiring in August 2027", QueryIntent.METADATA_QUERY),
    ("Which certificates have a missing expiration date?", QueryIntent.METADATA_QUERY),
    ("Show the expiration date for model RAD-992-X", QueryIntent.METADATA_QUERY),
    ("List all certificates issued in France", QueryIntent.METADATA_QUERY),

    # =========================================================================
    # --- UNSTRUCTURED_RAG Benchmark (33 Queries) ---
    # =========================================================================
    ("What are the test requirements for section 4?", QueryIntent.UNSTRUCTURED_RAG),
    ("Explain the quality management policy details", QueryIntent.UNSTRUCTURED_RAG),
    ("What emissions limits are specified for cold starts?", QueryIntent.UNSTRUCTURED_RAG),
    ("What is the maximum allowed EIRP transmit power for the 24GHz band?", QueryIntent.UNSTRUCTURED_RAG),
    ("Describe the out-of-band emission test procedure", QueryIntent.UNSTRUCTURED_RAG),
    ("What environmental temperature range was used during thermal testing?", QueryIntent.UNSTRUCTURED_RAG),
    ("Explain the antenna gain restrictions listed in the compliance report", QueryIntent.UNSTRUCTURED_RAG),
    ("What human RF exposure limits are cited in the documentation?", QueryIntent.UNSTRUCTURED_RAG),
    ("What are the operational safety precautions for installing the radar module?", QueryIntent.UNSTRUCTURED_RAG),
    ("Describe the radiated spurious emissions measurement setup", QueryIntent.UNSTRUCTURED_RAG),
    ("What modulation scheme was tested during the 5.8GHz spectrum evaluation?", QueryIntent.UNSTRUCTURED_RAG),
    ("Explain the duty cycle restrictions outlined in Section 3.2", QueryIntent.UNSTRUCTURED_RAG),
    ("What specific standard was referenced for ESD immunity testing?", QueryIntent.UNSTRUCTURED_RAG),
    ("Detail the occupying bandwidth requirements mentioned in the text", QueryIntent.UNSTRUCTURED_RAG),
    ("What were the failure criteria during the vibration stress test?", QueryIntent.UNSTRUCTURED_RAG),
    ("How is the device required to handle co-channel radio interference?", QueryIntent.UNSTRUCTURED_RAG),
    ("Describe the co-location restrictions regarding secondary transmitters", QueryIntent.UNSTRUCTURED_RAG),
    ("What frequency tolerance limit is mandated across operating temperatures?", QueryIntent.UNSTRUCTURED_RAG),
    ("Explain the testing methodology for the Bluetooth Low Energy module", QueryIntent.UNSTRUCTURED_RAG),
    ("What conductive emissions limits apply between 150kHz and 30MHz?", QueryIntent.UNSTRUCTURED_RAG),
    ("Summarize the modification policy regarding hardware changes post-approval", QueryIntent.UNSTRUCTURED_RAG),
    ("What labeling requirements are mandated on the exterior chassis?", QueryIntent.UNSTRUCTURED_RAG),
    ("Explain the antenna port conducted measurement procedure", QueryIntent.UNSTRUCTURED_RAG),
    ("What shielding requirements are specified for the RF transceiver enclosure?", QueryIntent.UNSTRUCTURED_RAG),
    ("Describe the test conditions during the salt spray corrosion assessment", QueryIntent.UNSTRUCTURED_RAG),
    ("What power spectral density limits are defined in Clause 5.1?", QueryIntent.UNSTRUCTURED_RAG),
    ("Explain how spurious responses are mitigated in the receiver circuit", QueryIntent.UNSTRUCTURED_RAG),
    ("What are the SAR testing exemption criteria listed in the report?", QueryIntent.UNSTRUCTURED_RAG),
    ("Describe the harmonic emission testing procedures for radar signals", QueryIntent.UNSTRUCTURED_RAG),
    ("What transient immunity test levels were applied to the DC power leads?", QueryIntent.UNSTRUCTURED_RAG),
    ("Explain the manufacturer's statement regarding software security updates", QueryIntent.UNSTRUCTURED_RAG),
    ("What beamforming constraints are noted for the phased array antenna?", QueryIntent.UNSTRUCTURED_RAG),
    ("Summarize the user manual safety warning requirements for RF hazard", QueryIntent.UNSTRUCTURED_RAG),

    # =========================================================================
    # --- HYBRID_QUERY Benchmark (33 Queries) ---
    # =========================================================================
    ("For German certificates, what is the warranty policy?", QueryIntent.HYBRID_QUERY),
    ("What are the compliance test requirements for Bosch model X?", QueryIntent.HYBRID_QUERY),
    ("Summarize safety guidelines for certificates expiring in 2026", QueryIntent.HYBRID_QUERY),
    ("What transmit power limits are specified for certificates issued in Japan?", QueryIntent.HYBRID_QUERY),
    ("For Continental certificates, explain the radiated emissions test setup", QueryIntent.HYBRID_QUERY),
    ("What antenna gain restrictions apply to devices certified by TÜV SÜD?", QueryIntent.HYBRID_QUERY),
    ("Summarize the ESD immunity test limits for certificates issued in 2024", QueryIntent.HYBRID_QUERY),
    ("For certificates expiring after 2027, what are the labeling requirements?", QueryIntent.HYBRID_QUERY),
    ("What operating frequency bands are declared for Denso certificates in Brazil?", QueryIntent.HYBRID_QUERY),
    ("Explain the out-of-band emission limits for certificates granted by FCC", QueryIntent.HYBRID_QUERY),
    ("What temperature testing range was applied to Valeo modules certified in France?", QueryIntent.HYBRID_QUERY),
    ("For certificates issued to Aptiv, summarize the RF human exposure warnings", QueryIntent.HYBRID_QUERY),
    ("What modulation schemes are approved for certificates issued in South Korea?", QueryIntent.HYBRID_QUERY),
    ("For model RF-7700, what are the duty cycle limitations?", QueryIntent.HYBRID_QUERY),
    ("Summarize the vibration test parameters for certificates issued in 2023", QueryIntent.HYBRID_QUERY),
    ("What shielding requirements apply to 77GHz radar certificates issued by ISED?", QueryIntent.HYBRID_QUERY),
    ("For German certificates, what spurious emission limits are enforced below 1GHz?", QueryIntent.HYBRID_QUERY),
    ("What are the SAR limits for handheld devices certified under TELEC Japan?", QueryIntent.HYBRID_QUERY),
    ("For certificates issued by Dekra, explain the conducted disturbance limits", QueryIntent.HYBRID_QUERY),
    ("What antenna co-location constraints are listed for Bosch certificates?", QueryIntent.HYBRID_QUERY),
    ("Summarize the transient immunity test results for certificates expiring in 2025", QueryIntent.HYBRID_QUERY),
    ("For certificates issued in the United States, what user manual warnings are mandatory?", QueryIntent.HYBRID_QUERY),
    ("What occupy bandwidth constraints apply to Continental model RAD-992-X?", QueryIntent.HYBRID_QUERY),
    ("For certificates approved between 2020 and 2022, explain the hardware modification rules?", QueryIntent.HYBRID_QUERY),
    ("What power spectral density limits are enforced for certificates issued by Bureau Veritas?", QueryIntent.HYBRID_QUERY),
    ("For certificates granted to Denso, what fail-safe mechanisms are required?", QueryIntent.HYBRID_QUERY),
    ("Summarize the ambient noise conditions during testing for UK certificates", QueryIntent.HYBRID_QUERY),
    ("For certificates expiring in 2028, what thermal shock protocols were performed?", QueryIntent.HYBRID_QUERY),
    ("What frequency stability thresholds apply to certificates issued in Canada?", QueryIntent.HYBRID_QUERY),
    ("For certificates issued to Valeo, what harmonic suppression limits are specified?", QueryIntent.HYBRID_QUERY),
    ("Explain the voltage fluctuation test requirements for certificates approved in 2025", QueryIntent.HYBRID_QUERY),
    ("For model UWB-001, what maximum EIRP output is legally authorized?", QueryIntent.HYBRID_QUERY),
    ("What EMC testing standards apply to certificates issued by SGS in Germany?", QueryIntent.HYBRID_QUERY),
]

# 3. Run Benchmark Evaluation Loop

print("=" * 85)
print("🚀 RUNNING EMPIRICAL BENCHMARK TEST (Qwen3-35B Engine, Non-Thinking Mode, max_tokens=4096)")
print("=" * 85)

correct_count = 0
total_time_ms = 0
benchmark_results = []

for idx, (query_text, expected_intent) in enumerate(benchmark_queries, 1):
    start_time = time.perf_counter()
    
    # Classify query intent using router
    result = classify_intent(query_text)
    
    latency_ms = (time.perf_counter() - start_time) * 1000
    total_time_ms += latency_ms
    
    predicted_intent_str = result.get("intent", "UNKNOWN")
    reasoning = result.get("reasoning", "N/A")
    expected_intent_str = expected_intent.value
    
    is_correct = (predicted_intent_str == expected_intent_str)
    if is_correct:
        correct_count += 1
        status_icon = "✅ MATCH"
    else:
        status_icon = "❌ MISMATCH"

    benchmark_results.append({
        "Test #": idx,
        "Status": status_icon,
        "Query": query_text,
        "Expected Intent": expected_intent_str,
        "Predicted Intent": predicted_intent_str,
        "Latency (ms)": round(latency_ms, 1),
        "Reasoning": reasoning
    })

    if idx % 10 == 0 or idx == len(benchmark_queries):
        current_acc = (correct_count / idx) * 100
        print(f"  [Progress {idx}/{len(benchmark_queries)}] Current Accuracy: {current_acc:.1f}% ({correct_count}/{idx}) | Avg Latency: {total_time_ms / idx:.1f} ms")

# 4. Display Benchmark Metrics Text
total_queries = len(benchmark_queries)
accuracy = (correct_count / total_queries) * 100
avg_latency_ms = total_time_ms / total_queries
total_time_sec = total_time_ms / 1000

print("\n" + "=" * 85)
print("📊 BENCHMARK SUMMARY REPORT — Qwen3-35B Engine (Non-Thinking Mode, max_tokens=4096)")
print("=" * 85)
print(f"  • Total Evaluation Time:        {total_time_sec:.2f} s ({total_time_ms:.1f} ms)")
print(f"  • Router Accuracy Rate:         {accuracy:.1f}% ({correct_count}/{total_queries} correct)")
print(f"  • Time Spent Per Query (Avg):   {avg_latency_ms:.1f} ms")
print("=" * 85 + "\n")

# 5. Display Final Results Table
import pandas as pd
from IPython.display import display
df_results = pd.DataFrame(benchmark_results)
display(df_results)


In [ ]:
# ==========================================
# 0. WORKSPACE UNPACK & SYNC
# ==========================================
# Purges pre-existing unpacked files at /content/Project before extracting project_sync.zip
import os, shutil, zipfile

zip_locations = ["/content/project_sync.zip", "/content/Project/project_sync.zip"]
found_zip = None
for z in zip_locations:
    if os.path.exists(z):
        found_zip = z
        break

if found_zip:
    print(f"📦 Found archive '{found_zip}'. Purging old unpacked workspace...")
    if os.path.exists("/content/Project"):
        shutil.rmtree("/content/Project")
    os.makedirs("/content/Project", exist_ok=True)
    
    print(f"📦 Extracting '{found_zip}' with full path normalization...")
    with zipfile.ZipFile(found_zip, 'r') as zip_ref:
        for member in zip_ref.infolist():
            clean_rel_path = member.filename.replace('\\', '/')
            if not clean_rel_path or clean_rel_path == '.':
                continue
            dest_filepath = os.path.join("/content/Project", clean_rel_path)
            if member.is_dir() or clean_rel_path.endswith('/'):
                os.makedirs(dest_filepath, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(dest_filepath), exist_ok=True)
                with zip_ref.open(member) as source, open(dest_filepath, "wb") as target:
                    target.write(source.read())
    print("✅ Successfully unpacked fresh workspace to /content/Project/\n")

# ==========================================
# 1. INITIALIZATION & SETUP
# ==========================================
# This script is the "clear event" that initializes everything:
# packages, models, llama-cpp, postgres database, and vLLM.
# It runs setup.sh and ensures the environment is strictly verified.
!bash /content/Project/setup.sh

# ==========================================
# 2. VERIFICATION (Subprocess to avoid kernel cache)
# ==========================================
import os, sys, subprocess, time, re

print("\n🔍 Running Pre-flight Environment Verification...")
verify_script = """
import sys
sys.path.insert(0, "/content/Project")
from colab.verify_env import verify_environment
is_ok, issues = verify_environment()
if is_ok:
    print("ALL_GOOD")
else:
    for issue in issues:
        print("ISSUE:", issue)
    sys.exit(1)
"""
with open("/tmp/verify.py", "w") as f:
    f.write(verify_script)

res = subprocess.run([sys.executable, "/tmp/verify.py"], capture_output=True, text=True)
if "ALL_GOOD" in res.stdout:
    print("\n✅ PRE-FLIGHT CHECKS PASSED: Environment fully initialized and verified!")
else:
    print("\n⚠️ PRE-FLIGHT CHECK ISSUES FOUND:")
    print(res.stdout)
    print(res.stderr)
    raise RuntimeError("Verification failed.")

# ==========================================
# 3. STREAMLIT LAUNCHER
# ==========================================
print("\n🚀 Launching Streamlit & Cloudflared...")
os.system("pkill -9 -f streamlit || true; pkill -9 -f cloudflared || true")
time.sleep(1)

os.makedirs("/content/Project/.streamlit", exist_ok=True)
with open("/content/Project/.streamlit/config.toml", "w") as f:
    f.write("[server]\nheadless = true\nenableCORS = false\nenableXsrfProtection = false\n")

st_log = open("/content/streamlit.log", "w")
st_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "/content/Project/ui/app.py",
     "--server.headless", "true", "--server.port", "8501"],
    cwd="/content/Project", stdout=st_log, stderr=st_log, start_new_session=True
)

if not os.path.exists("/content/cloudflared"):
    os.system("wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared")
    os.system("chmod +x /content/cloudflared")

cf_log = open("/content/cloudflared.log", "w")
cf_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=cf_log, stderr=cf_log, start_new_session=True
)

tunnel_url = None
t_end = time.time() + 20
while time.time() < t_end:
    time.sleep(1)
    if os.path.exists("/content/cloudflared.log"):
        with open("/content/cloudflared.log") as f:
            content = f.read()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    print("\n" + "=" * 70)
    print(f"🎉 STREAMLIT PUBLIC URL: {tunnel_url}")
    print("=" * 70)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🚀 RF COMPLIANCE PLATFORM — Dual-Tunnel Launcher (React UI + FastAPI)
# Port 8000: FastAPI backend  → Cloudflare Tunnel A (API)
# Port 8080: React static UI  → Cloudflare Tunnel B (opens as real browser tab)
# ═══════════════════════════════════════════════════════════════
import os, sys, re, time, subprocess, zipfile, threading
from http.server import HTTPServer, SimpleHTTPRequestHandler

# ── 1. Unpack project zip ────────────────────────────────────────
zip_path = "/content/project_sync.zip"
if not os.path.exists(zip_path):
    for f in os.listdir("/content"):
        if f.endswith(".zip"):
            zip_path = os.path.join("/content", f)
            break
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content/Project")
    print("✅ Unpacked project_sync.zip")
else:
    print("⚠️ No zip found — using existing /content/Project files")

os.environ["HF_HOME"] = "/content/model_cache"
if "/content/Project" not in sys.path:
    sys.path.insert(0, "/content/Project")

# ── 2. Patch CORS into main.py (idempotent) ─────────────────────
main_path = "/content/Project/main.py"
with open(main_path) as f:
    src = f.read()
if "CORSMiddleware" not in src:
    cors_block = '''
from fastapi.middleware.cors import CORSMiddleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
'''
    src = re.sub(r'(app\s*=\s*FastAPI\([^)]*\))', r'\1' + cors_block, src, flags=re.DOTALL)
    with open(main_path, "w") as f:
        f.write(src)
    print("✅ CORS patched into main.py")
else:
    print("ℹ️ CORS already in main.py")

# ── 3. Kill old servers ─────────────────────────────────────────
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
subprocess.run(["fuser", "-k", "8080/tcp"], capture_output=True)
time.sleep(1)

# ── 4. Launch FastAPI backend on port 8000 ────────────────────────
print("🚀 Starting FastAPI on port 8000...")
subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "warning"],
    cwd="/content/Project",
    stdout=open("/tmp/uvicorn.log", "w"), stderr=subprocess.STDOUT
)
time.sleep(4)
try:
    import urllib.request
    urllib.request.urlopen("http://localhost:8000/", timeout=5)
    print("✅ FastAPI backend live")
except Exception as e:
    print(f"⚠️ {e}")

# ── 5. Download cloudflared once ────────────────────────────────────
if not os.path.exists("/tmp/cloudflared"):
    print("⬇️ Downloading cloudflared...")
    subprocess.run(["curl", "-s", "-L",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-o", "/tmp/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/tmp/cloudflared"])

def get_tunnel_url(log_file, timeout=24):
    """Wait for cloudflared to print its public URL."""
    for _ in range(timeout):
        time.sleep(2)
        try:
            with open(log_file) as f:
                urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', f.read())
            if urls:
                return urls[0]
        except:
            pass
    return None

# ── 6. Tunnel A: expose FastAPI backend (port 8000) ──────────────────
print("🌐 Starting Cloudflare tunnel for API (port 8000)...")
subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=open("/tmp/cfd_api.log", "w"), stderr=subprocess.STDOUT)

BACKEND_URL = get_tunnel_url("/tmp/cfd_api.log")
if not BACKEND_URL:
    print("❌ Could not get API tunnel URL"); raise SystemExit
print(f"✅ API tunnel: {BACKEND_URL}")

# ── 7. Inject backend URL into React HTML and stage it for serving ───────
react_src = "/content/Project/ui/react/index.html"
with open(react_src) as f:
    html = f.read()
html = html.replace(
    'const API_BASE = window.API_BASE_URL || "http://localhost:8000";',
    f'const API_BASE = "{BACKEND_URL}";'
)
os.makedirs("/tmp/react_ui", exist_ok=True)
with open("/tmp/react_ui/index.html", "w") as f:
    f.write(html)

# ── 8. Serve React HTML on port 8080 via Python's built-in HTTP server ───
class SilentHandler(SimpleHTTPRequestHandler):
    def log_message(self, *args): pass  # suppress noisy request logs
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory="/tmp/react_ui", **kwargs)

server = HTTPServer(("0.0.0.0", 8080), SilentHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("✅ React UI HTTP server running on port 8080")

# ── 9. Tunnel B: expose React UI (port 8080) ───────────────────────
print("🌐 Starting Cloudflare tunnel for UI (port 8080)...")
subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8080"],
    stdout=open("/tmp/cfd_ui.log", "w"), stderr=subprocess.STDOUT)

FRONTEND_URL = get_tunnel_url("/tmp/cfd_ui.log")
if not FRONTEND_URL:
    print("❌ Could not get UI tunnel URL"); raise SystemExit

print(f"""
╔══════════════════════════════════════════════════════════╗
║  🎉 RF COMPLIANCE PLATFORM — READY                       ║
╠══════════════════════════════════════════════════════════╣
║  🖥️  OPEN THIS IN YOUR BROWSER ↓                         ║
║  {FRONTEND_URL:<54}                                      ║
╠══════════════════════════════════════════════════════════╣
║  🔗 Backend API (internal): {BACKEND_URL:<31}            ║
╚══════════════════════════════════════════════════════════╝
""")


In [ ]:
import os
import sys
import time
import tempfile

# 1. Ensure project root is in Python path
PROJECT_ROOT = "/content/Project" if os.path.exists("/content/Project") else os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Force model downloads to fast local NVMe cache
os.environ["HF_HOME"] = "/content/model_cache/"

# 2. Imports from project architecture
from config import CACHE_DIR
from engines.utils.system_check import initialize_system
from engines.utils.model_lifecycle import load_ocr_only, unload_ocr, load_llm_only
from engines.extractor import extract_certificate_data, save_certificate_to_db

# 3. System & Database Initialization
print("🗄️ Initializing system runtime and PostgreSQL pgvector tables...")
initialize_system()

# 4. Target Document Configuration
target_file = "/content/AR_IM3C_ENACOM_ID_H-22392__04.04.2028_.pdf"

# Fallback check if path differs in your current session
if not os.path.exists(target_file):
    local_target = os.path.join(PROJECT_ROOT, "AR_IM3C_ENACOM_ID_H-22392__04.04.2028_.pdf")
    if os.path.exists(local_target):
        target_file = local_target
    else:
        # Pick any PDF in /content or fallback to instructions
        pdfs = [os.path.join("/content", f) for f in os.listdir("/content") if f.lower().endswith(".pdf")]
        if pdfs:
            target_file = pdfs[0]
        else:
            raise FileNotFoundError(
                f"Target certificate standard PDF missing at '{target_file}'. "
                "Please upload the PDF into Colab Files or adjust target_file path."
            )

file_name = os.path.basename(target_file)
output_folder = os.path.join(tempfile.gettempdir(), "ocr_benchmark_outputs")
os.makedirs(output_folder, exist_ok=True)

print(f"\n📄 Target Certificate: {file_name}")

# ==============================================================================
# PHASE 1: GLM-OCR Performance Verification
# ==============================================================================
print("\n🚀 Phase 1: Loading GLM-OCR Engine...")
t0 = time.time()
ocr_engine = load_ocr_only(CACHE_DIR)
ocr_load_time = time.time() - t0
print(f"⏱️ Model Load Time: {ocr_load_time:.2f} s")

print(f"\n🖼️ Executing Phase 1 OCR Inference...")
t1 = time.time()
extracted_text = ocr_engine.process_document(target_file, output_folder)
total_ocr_time = time.time() - t1

# Estimate page count from OCR tags
num_pages = extracted_text.count("<Page ") if "<Page " in extracted_text else 1
num_pages = max(1, num_pages)
avg_time_per_page = total_ocr_time / num_pages
target_met = "✅ MET" if avg_time_per_page < 2.0 else "❌ EXCEEDED"

print("\n" + "=" * 70)
print(f"⏱️ Total OCR Time       : {total_ocr_time:.2f} s")
print(f"⏱️ Average Time Per Page: {avg_time_per_page:.2f} s/page")
print(f"⏱️ Target Ceiling (< 2s) : {target_met}")
print("=" * 70)

print("\n📝 Raw OCR Markdown Output Preview (first 500 chars):")
print("-" * 50)
print(extracted_text[:500])
print("-" * 50)

# Unload OCR Engine to free VRAM for LLM (Anti-OOM Sequential Lifecycle)
print("\n🛑 Unloading OCR engine...")
unload_ocr(ocr_engine)
ocr_engine = None

# ==============================================================================
# PHASE 2: Gemma 4 26B GGUF Extraction & DB Persistence
# ==============================================================================
print("\n🧠 Phase 2: Loading Gemma 4 26B GGUF Engine...")
t2 = time.time()
load_llm_only()
llm_load_time = time.time() - t2
print(f"⏱️ LLM Load Time: {llm_load_time:.2f} s")

print("\n🔍 Running extraction on raw OCR output...")
t3 = time.time()
cert_data = extract_certificate_data(extracted_text, file_name)
extraction_time = time.time() - t3
print(f"⏱️ Extraction Execution Time: {extraction_time:.2f} s")

print("\n📊 Extracted Certificate Metadata:")
print(f"  - Component    : {cert_data.component}")
print(f"  - Supplier     : {cert_data.supplier}")
print(f"  - Country      : {cert_data.country}")
print(f"  - Certif Number: {cert_data.certif_number}")
print(f"  - Authority    : {cert_data.authority}")
print(f"  - Issue Date   : {cert_data.issue_date}")
print(f"  - Exp Date     : {cert_data.exp_date}")

# Save Metadata + 1024-d Embeddings to PostgreSQL
db_record = save_certificate_to_db(cert_data, extracted_text, file_name)
print(f"\n✅ Persisted to PostgreSQL DB! Record ID: {db_record.certificate_id}")
print("🎉 End-to-End Pipeline Execution Verified Successfully!")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🚀 RF COMPLIANCE PLATFORM — React + FastAPI One-Shot Launcher
# ═══════════════════════════════════════════════════════════════
# Dependencies already installed via colab/setup.sh (requirements.txt Step 6).
# This cell only: syncs files, patches CORS, launches uvicorn + cloudflared, renders UI.
import os, sys, re, time, subprocess, zipfile
from IPython.display import IFrame, display

# ── 1. Unpack project zip ────────────────────────────────────────
zip_path = "/content/project_sync.zip"
if not os.path.exists(zip_path):
    for f in os.listdir("/content"):
        if f.endswith(".zip"):
            zip_path = os.path.join("/content", f)
            break

if os.path.exists(zip_path):
    os.makedirs("/content/Project", exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content/Project")
    print("✅ Unpacked project_sync.zip to /content/Project/")
else:
    print("⚠️ No zip found — using existing /content/Project files")

os.environ["HF_HOME"] = "/content/model_cache"
if "/content/Project" not in sys.path:
    sys.path.insert(0, "/content/Project")

# ── 2. Patch CORS into main.py (idempotent) ─────────────────────
main_path = "/content/Project/main.py"
with open(main_path) as f:
    src = f.read()

if "CORSMiddleware" not in src:
    cors_block = '''
from fastapi.middleware.cors import CORSMiddleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
'''
    src = re.sub(
        r'(app\s*=\s*FastAPI\([^)]*\))',
        r'\1' + cors_block,
        src, flags=re.DOTALL
    )
    with open(main_path, "w") as f:
        f.write(src)
    print("✅ CORS middleware patched into main.py")
else:
    print("ℹ️ CORS already present in main.py — skipping")

# ── 3. Kill any old servers ───────────────────────────────────────
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(1)

# ── 4. Launch FastAPI on port 8000 ──────────────────────────────
print("🚀 Starting FastAPI backend on port 8000...")
subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "warning"],
    cwd="/content/Project",
    stdout=open("/tmp/uvicorn.log", "w"),
    stderr=subprocess.STDOUT
)
time.sleep(4)

try:
    import urllib.request
    urllib.request.urlopen("http://localhost:8000/", timeout=5)
    print("✅ FastAPI backend live at http://localhost:8000/")
except Exception as e:
    print(f"⚠️ Backend check: {e} — may still be initializing")

# ── 5. Expose port 8000 via Cloudflare Tunnel ───────────────────
if not os.path.exists("/tmp/cloudflared"):
    print("⬇️ Downloading cloudflared...")
    subprocess.run([
        "curl", "-s", "-L",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-o", "/tmp/cloudflared"
    ], check=True)
    subprocess.run(["chmod", "+x", "/tmp/cloudflared"])

subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=open("/tmp/cfd.log", "w"),
    stderr=subprocess.STDOUT
)
print("🌐 Waiting for Cloudflare tunnel URL...")

BACKEND_URL = None
for _ in range(12):
    time.sleep(2)
    try:
        with open("/tmp/cfd.log") as f:
            log = f.read()
        urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log)
        if urls:
            BACKEND_URL = urls[0]
            break
    except:
        pass

if not BACKEND_URL:
    print("❌ Could not get Cloudflare URL. Check /tmp/cfd.log")
    raise SystemExit

print(f"✅ Backend public URL: {BACKEND_URL}")

# ── 6. Inject backend URL into React app & display inside Colab ──
react_path = "/content/Project/ui/react/index.html"
with open(react_path) as f:
    html = f.read()

html = html.replace(
    'const API_BASE = window.API_BASE_URL || "http://localhost:8000";',
    f'const API_BASE = "{BACKEND_URL}";'
)

out_path = "/tmp/react_app.html"
with open(out_path, "w") as f:
    f.write(html)

print(f"""
══════════════════════════════════════════════════
🎉 RF COMPLIANCE PLATFORM — React + FastAPI
🔗 Backend API  : {BACKEND_URL}
🖥️  Frontend UI : Rendered below in Colab output
══════════════════════════════════════════════════
""")

# Render React app natively in Colab output — no tunnel needed for the UI!
display(IFrame(src=out_path, width="100%", height="800"))
